# LeetCode 672 — Bulb Switcher II bằng Python

Link đề: https://leetcode.com/problems/bulb-switcher-ii/ · Độ khó: **Medium** · Chủ đề: Math, Bit Manipulation (thao tác bit), BFS/DFS, XOR trên các trạng thái

Skeleton LeetCode:

```python
class Solution:
    def flipLights(self, n: int, presses: int) -> int:
        ...
```

Ghi chú: `n` và `presses` tới `1000` nhưng đáp án **chỉ có tối đa 8**. Bài này là bài "tìm quy luật": mô phỏng được ở quy mô nhỏ để quan sát, rồi chứng minh vì sao quy luật đúng (nhấn hai lần = huỷ nhau; bóng `j` và `j + 6` luôn giống nhau; nút 1 = nút 2 XOR nút 3). Notebook đi từ BFS mô phỏng đầy đủ, BFS rút gọn bằng bitmask 6 bóng, liệt kê tập nút theo tính chẵn lẻ, đến công thức theo trường hợp.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: `n` bóng đèn ban đầu **bật**, 4 nút với 4 kiểu bật/tắt (đảo trạng thái) khác nhau; phải bấm **đúng** `presses` lần (mỗi lần chọn nút tuỳ ý); đếm số **trạng thái khác nhau** có thể có.
2. Nắm các công cụ Python/bit cần dùng: phép **XOR** `^` (đảo bit), bitmask, `set` để loại trùng, `bin`/`format` để in nhị phân, `itertools.product`, đếm bit (`int.bit_count()`), duyệt tập con bằng mask `0..15`.
3. Đi từ **BFS mô phỏng đủ `n` bóng** → **BFS với 6 bóng đầu bằng bitmask** → **liệt kê 16 tập nút theo tính chẵn lẻ** → **công thức theo trường hợp**, và hiểu vì sao các cách cho cùng đáp án.
4. Hiểu mấu chốt: (a) bấm một nút **hai lần** thì huỷ nhau nên chỉ **tính chẵn/lẻ** của số lần bấm mỗi nút quan trọng; (b) mọi nút chỉ phụ thuộc `j mod 6` nên trạng thái bóng `j` luôn bằng bóng `j + 6`; (c) nút 1 bằng nút 2 kết hợp nút 3, nên chỉ có tối đa `2^3 = 8` trạng thái.
5. Tránh các lỗi kinh điển: mô phỏng `n = 1000` quá chậm, nghĩ có `2^4 = 16` trạng thái, quên `presses = 0`, dùng bảng của `n ≥ 3` cho `n = 1, 2`, bỏ qua điều kiện chẵn lẻ của số lần bấm.

## 2. Đề bài & ví dụ

Trong phòng có `n` bóng đèn đánh số từ `1` đến `n`, ban đầu tất cả đều **bật**, và có **bốn nút** trên tường:

- **Nút 1:** đảo trạng thái của **tất cả** các bóng.
- **Nút 2:** đảo trạng thái các bóng có nhãn **chẵn** (`2, 4, ...`).
- **Nút 3:** đảo trạng thái các bóng có nhãn **lẻ** (`1, 3, ...`).
- **Nút 4:** đảo trạng thái các bóng có nhãn `j = 3k + 1` với `k = 0, 1, 2, ...` (tức `1, 4, 7, 10, ...`).

Bạn phải bấm **đúng** `presses` lần (mỗi lần chọn một trong bốn nút tuỳ ý). Cho hai số nguyên `n` và `presses`, trả về **số trạng thái khác nhau** của các bóng sau khi bấm xong.

**Ví dụ 1:** `n = 1, presses = 1` → `2` (bóng tắt khi bấm nút 1; bóng vẫn bật khi bấm nút 2).

**Ví dụ 2:** `n = 2, presses = 1` → `3`: `[tắt, tắt]` (nút 1), `[bật, tắt]` (nút 2), `[tắt, bật]` (nút 3).

**Ví dụ 3:** `n = 3, presses = 1` → `4`: `[tắt, tắt, tắt]` (nút 1), `[bật, tắt, bật]` (nút 2), `[tắt, bật, tắt]` (nút 3), `[tắt, bật, bật]` (nút 4).

**Ràng buộc:**

- `1 <= n <= 1000`
- `0 <= presses <= 1000`

Lưu ý "đúng `presses` lần" (không phải "tối đa"), và đếm **trạng thái** (dãy bật/tắt của `n` bóng), không đếm dãy bấm.

In [1]:
from typing import List
from itertools import product
import random

examples = [
    (1, 1, 2),
    (2, 1, 3),
    (3, 1, 4),
]
for n, presses, expected in examples:
    print(f"n = {n}, presses = {presses} -> kỳ vọng {expected}")

n = 1, presses = 1 -> kỳ vọng 2
n = 2, presses = 1 -> kỳ vọng 3
n = 3, presses = 1 -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1. Phép XOR `^` và "đảo bit"

`a ^ b` (XOR) so từng bit: hai bit **khác nhau** cho `1`, **giống nhau** cho `0`. Hệ quả hữu ích:

- `x ^ 1` đảo bit thấp nhất của `x` (0 thành 1, 1 thành 0).
- `x ^ m` đảo mọi bit của `x` ở vị trí mà mặt nạ `m` có bit `1`.
- **Đảo hai lần = trả lại cũ**: `(x ^ m) ^ m == x`.
- XOR có tính giao hoán, kết hợp: `a ^ b ^ c` không phụ thuộc thứ tự.

Chính "đảo hai lần = trả lại cũ" là lý do bấm một nút hai lần thì huỷ nhau.

In [2]:
x = 0b1011
m = 0b0110
print(format(x, "04b"), "^", format(m, "04b"), "=", format(x ^ m, "04b"))
print((x ^ m) ^ m == x)                 # True: đảo hai lần trả lại cũ
print(format(0b101 ^ 0b011, "03b"), format(0b011 ^ 0b101, "03b"))   # giao hoán
print(0 ^ 1, 1 ^ 1)                      # 1 0

1011 ^ 0110 = 1101
True
110 110
1 0


### 3.2. Bitmask: biểu diễn trạng thái dãy bật/tắt bằng một số nguyên

Một dãy `m` bóng (bật = `1`, tắt = `0`) có thể lưu trong **một số nguyên**: bóng thứ `j` (đếm từ 1) là bit số `j - 1`. Ví dụ 3 bóng, `[bật, tắt, bật]` là `0b101`. Khi đó "nút X đảo một nhóm bóng" chỉ là `state ^ mask_X`, với `mask_X` có bit `1` ở các bóng của nhóm.

Lợi ích: một trạng thái là một số, đưa được vào `set` nhanh gọn để loại trùng.

In [3]:
def to_mask(lights):
    # lights: list[bool], True = bật; bóng thứ j (từ 1) là bit j-1
    return sum(1 << i for i, on in enumerate(lights) if on)

def to_lights(mask, m):
    return [bool((mask >> i) & 1) for i in range(m)]

state = to_mask([True, False, True])
print(state, format(state, "03b"))                 # 5 101
print(to_lights(state, 3))
print(to_lights(state ^ 0b111, 3))                 # đảo cả 3 bóng

5 101
[True, False, True]
[False, True, False]


### 3.3. `set` để loại trùng và `int.bit_count()`

`set` chỉ giữ các phần tử **khác nhau**: thêm phần tử đã có thì không làm gì. `len(s)` cho số phần tử khác nhau — đúng thứ ta cần đếm ("số trạng thái khác nhau").

`int.bit_count()` (Python 3.10+) đếm số bit `1`: dùng để biết một tập nút (biểu diễn bằng mask 4 bit) có mấy nút.

In [4]:
s = set()
for st in [5, 3, 5, 7, 3]:
    s.add(st)
print(s, len(s))                                   # {3, 5, 7} 3
print((0b1011).bit_count(), (0b0000).bit_count())  # 3 0
print(bin(13), format(13, "04b"))

{3, 5, 7} 3
3 0
0b1101 1101


### 3.4. Duyệt mọi tập con của 4 phần tử bằng mask `0..15`

Với 4 nút, mỗi tập nút là một số `mask` từ `0` đến `15` (`2^4 = 16` tập). Bit `b` của `mask` bằng `1` nghĩa là nút `b + 1` có trong tập. Ví dụ `mask = 0b0101` là tập `{nút 1, nút 3}`.

In [5]:
for mask in range(16):
    buttons = [b + 1 for b in range(4) if (mask >> b) & 1]
    print(f"{mask:2d} = {mask:04b} -> nút {buttons}  (có {mask.bit_count()} nút)")

 0 = 0000 -> nút []  (có 0 nút)
 1 = 0001 -> nút [1]  (có 1 nút)
 2 = 0010 -> nút [2]  (có 1 nút)
 3 = 0011 -> nút [1, 2]  (có 2 nút)
 4 = 0100 -> nút [3]  (có 1 nút)
 5 = 0101 -> nút [1, 3]  (có 2 nút)
 6 = 0110 -> nút [2, 3]  (có 2 nút)
 7 = 0111 -> nút [1, 2, 3]  (có 3 nút)
 8 = 1000 -> nút [4]  (có 1 nút)
 9 = 1001 -> nút [1, 4]  (có 2 nút)
10 = 1010 -> nút [2, 4]  (có 2 nút)
11 = 1011 -> nút [1, 2, 4]  (có 3 nút)
12 = 1100 -> nút [3, 4]  (có 2 nút)
13 = 1101 -> nút [1, 3, 4]  (có 3 nút)
14 = 1110 -> nút [2, 3, 4]  (có 3 nút)
15 = 1111 -> nút [1, 2, 3, 4]  (có 4 nút)


### 3.5. `itertools.product` để liệt kê mọi dãy bấm

`product(range(4), repeat=k)` sinh mọi dãy `k` lần bấm, mỗi lần chọn nút `0..3`: có `4^k` dãy. Ta dùng nó như **oracle thô** ở quy mô rất nhỏ (`k ≤ 8`), chuyên để kiểm tra các cách thông minh.

In [6]:
print(list(product(range(2), repeat=2)))
print(len(list(product(range(4), repeat=5))), "== 4**5 =", 4**5)

[(0, 0), (0, 1), (1, 0), (1, 1)]
1024 == 4**5 = 1024


## 4. Ý tưởng nền tảng / trực giác

Tính tay `n = 3`, bóng `1, 2, 3` đều bật. Các nút trên 3 bóng:

- Nút 1: đảo `{1, 2, 3}`.
- Nút 2: đảo `{2}` (nhãn chẵn).
- Nút 3: đảo `{1, 3}` (nhãn lẻ).
- Nút 4: đảo `{1}` (nhãn `3k+1` bằng `1`; `4 > 3`).

**Nhận xét 1:** bấm nút 2 hai lần = không bấm. Do đó, dù bấm tổng cộng bao nhiêu lần, chỉ **tính chẵn/lẻ số lần bấm từng nút** mới quyết định trạng thái cuối. Có `2^4 = 16` mẫu chẵn/lẻ.

**Nhận xét 2:** nút 1 đảo mọi bóng, nút 2 đảo bóng chẵn, nút 3 đảo bóng lẻ. Bấm nút 2 rồi nút 3 đảo **mọi** bóng — đúng như nút 1. Vậy "bấm 1, 2, 3 cùng lúc" = không làm gì cả. Số trạng thái thật sự khác nhau **ít hơn 16**.

Cả hai nhận xét sẽ được kiểm chứng bằng code ở các mục sau.

In [7]:
# 3 bóng: mỗi nút là một mặt nạ bit (bóng j là bit j-1)
n = 3
all_on = (1 << n) - 1
b1 = 0b111          # đảo tất cả
b2 = 0b010          # bóng 2
b3 = 0b101          # bóng 1 và 3
b4 = 0b001          # bóng 1

print("đảo hai lần trả lại cũ:", (all_on ^ b2 ^ b2) == all_on)
print("b2 ^ b3 == b1:", (b2 ^ b3) == b1)               # nút 2 + nút 3 = nút 1
print("b1 ^ b2 ^ b3 == 0:", (b1 ^ b2 ^ b3) == 0)       # bấm cả 3 nút = không làm gì

đảo hai lần trả lại cũ: True
b2 ^ b3 == b1: True
b1 ^ b2 ^ b3 == 0: True


## 5. Cách 1 — Brute Force: BFS mô phỏng đủ `n` bóng

### Ý tưởng

Dùng BFS theo **từng lần bấm**:

1. Tập `cur` chứa các trạng thái có thể có sau `t` lần bấm; ban đầu `cur = {tất cả bật}`.
2. Mỗi lần bấm: với mỗi trạng thái trong `cur` và mỗi nút trong 4 nút, tạo trạng thái mới; gom vào `nxt` (một `set` để loại trùng).
3. Sau `presses` lần, trả về `len(cur)`.

Trạng thái là `tuple` các giá trị `True/False` của đủ `n` bóng, để thấy rõ nhất cách đề mô tả. Ta cài hàm "nút `b` áp dụng lên bóng nhãn `j`" đúng theo đề.

### Code

In [8]:
def applies(button, j):
    # nút (1..4) có đảo bóng nhãn j (j từ 1) không?
    if button == 1:
        return True
    if button == 2:
        return j % 2 == 0
    if button == 3:
        return j % 2 == 1
    if button == 4:
        return j % 3 == 1
    raise ValueError(button)

def press(state, button):
    return tuple((not on) if applies(button, j + 1) else on for j, on in enumerate(state))

def flip_lights_bfs_full(n, presses):
    cur = {tuple([True] * n)}
    for _ in range(presses):
        cur = {press(st, b) for st in cur for b in (1, 2, 3, 4)}
    return len(cur)

for n, presses, expected in examples:
    got = flip_lights_bfs_full(n, presses)
    print(got, expected, "OK" if got == expected else "SAI")

2 2 OK
3 3 OK
4 4 OK


### Độ phức tạp và nút thắt

- **Time:** `O(presses · |trạng thái| · 4 · n)`. Số trạng thái khác nhau tối đa là 8 nên mỗi lần bấm chỉ xử lý vài trạng thái; chi phí mỗi trạng thái `O(n)`. Với `n = presses = 1000` là khoảng `1000 · 8 · 4 · 1000 = 32 triệu` thao tác Python — chậm (hàng chục giây).
- **Space:** `O(8 · n)` cho tập trạng thái.

**Nút thắt:** (1) ta cứ mô phỏng đủ `n` bóng dù thực chất các bóng lặp lại; (2) vẫn lặp `presses` lần dù tập trạng thái đã ổn định. Hai quan sát ở mục 6 loại bỏ cả hai. Cách này vẫn quý: nó là **oracle trung thực nhất** cho các cách nhanh hơn (chạy với `n, presses` nhỏ).

In [9]:
# Xem tập trạng thái khi n = 3 qua từng lần bấm: lớn dần rồi ổn định ở 8
for presses in range(0, 7):
    print(f"presses = {presses}: {flip_lights_bfs_full(3, presses)} trạng thái")

presses = 0: 1 trạng thái
presses = 1: 4 trạng thái
presses = 2: 7 trạng thái
presses = 3: 8 trạng thái
presses = 4: 8 trạng thái
presses = 5: 8 trạng thái
presses = 6: 8 trạng thái


## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát A — Chu kỳ 6.** Cả bốn nút chỉ phụ thuộc `j mod 2` hoặc `j mod 3`:

- nút 1: luôn đảo; nút 2: `j mod 2 == 0`; nút 3: `j mod 2 == 1`; nút 4: `j mod 3 == 1`.

Vì `lcm(2, 3) = 6`, bóng `j` và bóng `j + 6` **luôn bị đảo cùng nhau** ở mọi lần bấm; ban đầu cùng bật, nên **luôn giống nhau**. Do đó trạng thái của `n` bóng **được xác định hoàn toàn bởi `min(n, 6)` bóng đầu**; số trạng thái của `n ≥ 6` bóng bằng số trạng thái của `6` bóng.

**Quan sát B — Chỉ cần chẵn/lẻ.** Nút nào cũng là phép đảo (XOR với một mặt nạ), nên bấm hai lần huỷ nhau; thứ tự không quan trọng. Một dãy `presses` lần bấm tương ứng với một **tập `S`** các nút (nút nào được bấm **số lẻ lần** thì thuộc `S`).

Điều kiện để một tập `S` đạt được với đúng `presses` lần bấm:

- `|S| ≤ presses` (mỗi nút trong `S` cần ít nhất 1 lần), và
- `presses - |S|` là số **chẵn** (số lần bấm còn thừa phải được dùng theo cặp: bấm một nút hai lần huỷ nhau).

Ngược lại, tập `S` nào thoả hai điều kiện đó đều đạt được. Vậy đáp án = số trạng thái khác nhau của `{ XOR các nút trong S : |S| ≤ presses, (presses - |S|) chẵn }`, tính trên `min(n, 6)` bóng đầu. Chỉ có 16 tập `S`.

Cell dưới kiểm chứng Quan sát A: với mọi dãy bấm ngẫu nhiên, bóng `j` và `j + 6` luôn giống nhau.

In [10]:
random.seed(672)
checked = 0
for _ in range(2000):
    n = random.randint(7, 20)
    st = tuple([True] * n)
    for _ in range(random.randint(0, 15)):
        st = press(st, random.randint(1, 4))
    for j in range(n - 6):
        assert st[j] == st[j + 6], (n, st)
    checked += 1
print("chu kỳ 6 đúng trên", checked, "dãy bấm ngẫu nhiên")

chu kỳ 6 đúng trên 2000 dãy bấm ngẫu nhiên


## 7. Các cách tiếp theo

### 7.1. Cách 2 — BFS rút gọn: chỉ 6 bóng đầu, trạng thái là bitmask

**Giải thích từng đoạn.**

1. `m = min(n, 6)`: chỉ giữ `m` bóng đầu (Quan sát A).
2. Mỗi nút thành một mặt nạ `m` bit (bit `j-1` là bóng `j`): ví dụ với `m = 6`: nút 1 `0b111111`, nút 2 (bóng 2,4,6) `0b101010`, nút 3 (bóng 1,3,5) `0b010101`, nút 4 (bóng 1,4) `0b001001`.
3. BFS như Cách 1 nhưng trạng thái là một số nguyên: `new = state ^ mask`. Có tối đa `2^6 = 64` trạng thái nên tập luôn nhỏ.

Cắt giảm: từ `O(n)` mỗi trạng thái xuống `O(1)`.

**Code demo:**

In [11]:
def make_masks(m):
    # mặt nạ 4 nút trên m bóng đầu; bóng j (từ 1) là bit j-1
    return [sum(1 << (j - 1) for j in range(1, m + 1) if applies(b, j)) for b in (1, 2, 3, 4)]

print([format(x, "06b") for x in make_masks(6)])    # nút 1..4 trên 6 bóng
print([format(x, "03b") for x in make_masks(3)])

def flip_lights_bfs_mask(n, presses):
    m = min(n, 6)
    masks = make_masks(m)
    cur = {(1 << m) - 1}                           # tất cả bật
    for _ in range(presses):
        cur = {st ^ mk for st in cur for mk in masks}
    return len(cur)

for n, presses, expected in examples:
    print(flip_lights_bfs_mask(n, presses), expected)
print(flip_lights_bfs_mask(1000, 1000))            # chạy nhanh

['111111', '101010', '010101', '001001']
['111', '010', '101', '001']
2 2
3 3
4 4
8


**Code nộp LeetCode (Cách 2):**

In [12]:
class SolutionBfsMask:
    def flipLights(self, n: int, presses: int) -> int:
        m = min(n, 6)
        full = (1 << m) - 1
        # mặt nạ trên 6 bóng (bit j-1 là bóng j), cắt về m bóng đầu
        ops = [0b111111 & full, 0b101010 & full, 0b010101 & full, 0b001001 & full]
        cur = {full}
        for _ in range(presses):
            cur = {st ^ op for st in cur for op in ops}
        return len(cur)

print(SolutionBfsMask().flipLights(3, 1))   # 4

4


**Độ phức tạp (Cách 2):** Time `O(presses · 64 · 4)` = `O(presses)` (hằng số nhỏ), Space `O(64)`. Có thể dừng sớm khi tập trạng thái lặp lại, nhưng không cần.

### 7.2. Cách 3 — Liệt kê 16 tập nút theo tính chẵn lẻ (lời giải nộp)

**Giải thích từng đoạn.**

1. Với mỗi `mask` trong `0..15` (tập nút `S`): `k = mask.bit_count()`.
2. Nếu `k ≤ presses` và `(presses - k) % 2 == 0` thì tập này đạt được (xem Quan sát B ở mục 6). Ngược lại bỏ qua.
3. Tính trạng thái: bắt đầu `full` (tất cả bật), `XOR` thêm mặt nạ của từng nút trong `S`. Bỏ trạng thái vào `set`.
4. Trả về `len(set)`.

Không còn vòng lặp theo `presses` — độ phức tạp **hằng số**.

**Code demo:**

In [13]:
def flip_lights_subsets(n, presses):
    m = min(n, 6)
    masks = make_masks(m)
    full = (1 << m) - 1
    seen = set()
    for sub in range(16):
        k = sub.bit_count()
        if k <= presses and (presses - k) % 2 == 0:
            st = full
            for b in range(4):
                if (sub >> b) & 1:
                    st ^= masks[b]
            seen.add(st)
    return len(seen)

for n, presses, expected in examples:
    print(flip_lights_subsets(n, presses), expected)
print(flip_lights_subsets(1000, 1000))

2 2
3 3
4 4
8


**Code nộp LeetCode (Cách 3):**

In [14]:
class SolutionSubsets:
    def flipLights(self, n: int, presses: int) -> int:
        m = min(n, 6)
        full = (1 << m) - 1
        ops = [0b111111 & full, 0b101010 & full, 0b010101 & full, 0b001001 & full]
        seen = set()
        for sub in range(16):                       # mỗi bit của sub: nút đó được bấm số lẻ lần
            k = sub.bit_count()
            if k <= presses and (presses - k) % 2 == 0:
                st = full
                for b in range(4):
                    if (sub >> b) & 1:
                        st ^= ops[b]
                seen.add(st)
        return len(seen)

print(SolutionSubsets().flipLights(2, 1))   # 3

3


**Độ phức tạp (Cách 3):** Time `O(16 · 4)` = `O(1)`, Space `O(8)` (tập trạng thái).

### 7.3. Cách 4 — Công thức theo trường hợp (suy ra từ cách 3)

**Giải thích từng đoạn.** Chạy Cách 3 để lập bảng đáp án theo `(n, presses)` rồi nhận ra bảng chỉ có vài trường hợp:

- `presses == 0`: không bấm gì, chỉ một trạng thái (tất cả bật) → `1`.
- `n == 1`: bóng bật/tắt → `2` (khi có bấm).
- `n == 2`: `presses == 1` → `3`; `presses ≥ 2` → `4`.
- `n ≥ 3`: `presses == 1` → `4`; `presses == 2` → `7`; `presses ≥ 3` → `8`.

Hàm dưới in bảng đó từ Cách 3 để bạn tự quan sát, rồi cài công thức.

In [15]:
print("presses ->  0  1  2  3  4  5")
for n in range(1, 9):
    print(f"n = {n}      " + "  ".join(str(flip_lights_subsets(n, p)) for p in range(6)))

presses ->  0  1  2  3  4  5
n = 1      1  2  2  2  2  2
n = 2      1  3  4  4  4  4
n = 3      1  4  7  8  8  8
n = 4      1  4  7  8  8  8
n = 5      1  4  7  8  8  8
n = 6      1  4  7  8  8  8
n = 7      1  4  7  8  8  8
n = 8      1  4  7  8  8  8


In [16]:
def flip_lights_formula(n, presses):
    if presses == 0:
        return 1
    if n == 1:
        return 2
    if n == 2:
        return 3 if presses == 1 else 4
    if presses == 1:
        return 4
    if presses == 2:
        return 7
    return 8

for n, presses, expected in examples:
    print(flip_lights_formula(n, presses), expected)

2 2
3 3
4 4


**Code nộp LeetCode (Cách 4):**

In [17]:
class SolutionFormula:
    def flipLights(self, n: int, presses: int) -> int:
        if presses == 0:
            return 1
        if n == 1:
            return 2
        if n == 2:
            return 3 if presses == 1 else 4
        if presses == 1:
            return 4
        return 7 if presses == 2 else 8

print(SolutionFormula().flipLights(3, 1))   # 4

4


**Độ phức tạp (Cách 4):** Time `O(1)`, Space `O(1)`.

Vì sao công thức đúng: xem mục 10 (đếm số tập `S` hợp lệ và số trạng thái khác nhau cho từng nhóm `n`).

## 8. TRACE từng bước

Trace Cách 3 cho `n = 3, presses = 1`. Mặt nạ trên 3 bóng: nút 1 = `111`, nút 2 = `010`, nút 3 = `101`, nút 4 = `001`. Tập `S` hợp lệ khi `|S| ≤ 1` và `1 - |S|` chẵn, tức `|S| = 1`: bốn tập `{1},{2},{3},{4}`.

| tập S | số nút | hợp lệ với presses = 1? | trạng thái dạng mặt nạ (bit cao nhất = bóng 3) | mới? |
|---|---|---|---|---|
| {} | 0 | không (1 - 0 lẻ) | — | — |
| {1} | 1 | có | `000` (tắt, tắt, tắt) | mới |
| {2} | 1 | có | `101` (bật, tắt, bật) | mới |
| {3} | 1 | có | `010` (tắt, bật, tắt) | mới |
| {4} | 1 | có | `110` (tắt, bật, bật) | mới |

4 trạng thái khác nhau → đáp án `4` (đúng Ví dụ 3). Hàm `trace_subsets` in bảng đầy đủ cho `(n, presses)` bất kỳ, gồm cả những tập bị loại.

In [18]:
def trace_subsets(n, presses):
    m = min(n, 6)
    masks = make_masks(m)
    full = (1 << m) - 1
    seen = {}
    print(f"n = {n} (dùng {m} bóng đầu), presses = {presses}; nút 1..4 = {[format(x, f'0{m}b') for x in masks]}")
    print(f"{'tập nút':<14}{'|S|':>4}  {'hợp lệ?':<22}{'trạng thái (bit j-1 = bóng j)':<32}ghi chú")
    for sub in range(16):
        buttons = [b + 1 for b in range(4) if (sub >> b) & 1]
        k = len(buttons)
        ok = k <= presses and (presses - k) % 2 == 0
        if not ok:
            reason = "k > presses" if k > presses else "presses - k lẻ"
            print(f"{str(buttons):<14}{k:>4}  {'không (' + reason + ')':<22}")
            continue
        st = full
        for b in buttons:
            st ^= masks[b - 1]
        note = "trùng với tập " + str(seen[st]) if st in seen else "MỚI"
        if st not in seen:
            seen[st] = buttons
        print(f"{str(buttons):<14}{k:>4}  {'có':<22}{format(st, f'0{m}b'):<32}{note}")
    print("số trạng thái khác nhau:", len(seen))
    return len(seen)

trace_subsets(3, 1)

n = 3 (dùng 3 bóng đầu), presses = 1; nút 1..4 = ['111', '010', '101', '001']
tập nút        |S|  hợp lệ?               trạng thái (bit j-1 = bóng j)   ghi chú
[]               0  không (presses - k lẻ)
[1]              1  có                    000                             MỚI
[2]              1  có                    101                             MỚI
[1, 2]           2  không (k > presses)   
[3]              1  có                    010                             MỚI
[1, 3]           2  không (k > presses)   
[2, 3]           2  không (k > presses)   
[1, 2, 3]        3  không (k > presses)   
[4]              1  có                    110                             MỚI
[1, 4]           2  không (k > presses)   
[2, 4]           2  không (k > presses)   
[1, 2, 4]        3  không (k > presses)   
[3, 4]           2  không (k > presses)   
[1, 3, 4]        3  không (k > presses)   
[2, 3, 4]        3  không (k > presses)   
[1, 2, 3, 4]     4  không (k > presses)   
số trạng thá

4

Trace thứ hai: `n = 3, presses = 3`. Các tập hợp lệ gồm `|S| = 1` (bấm nút đó 1 lần + một nút khác 2 lần) và `|S| = 3` (mỗi nút một lần). Chú ý tập `{1, 2, 3}` cho trạng thái **tất cả bật** (giống tập rỗng): đó là chỗ trùng và là lý do trạng thái ≤ 8 chứ không phải 16.

In [19]:
trace_subsets(3, 3)

n = 3 (dùng 3 bóng đầu), presses = 3; nút 1..4 = ['111', '010', '101', '001']
tập nút        |S|  hợp lệ?               trạng thái (bit j-1 = bóng j)   ghi chú
[]               0  không (presses - k lẻ)
[1]              1  có                    000                             MỚI
[2]              1  có                    101                             MỚI
[1, 2]           2  không (presses - k lẻ)
[3]              1  có                    010                             MỚI
[1, 3]           2  không (presses - k lẻ)
[2, 3]           2  không (presses - k lẻ)
[1, 2, 3]        3  có                    111                             MỚI
[4]              1  có                    110                             MỚI
[1, 4]           2  không (presses - k lẻ)
[2, 4]           2  không (presses - k lẻ)
[1, 2, 4]        3  có                    011                             MỚI
[3, 4]           2  không (presses - k lẻ)
[1, 3, 4]        3  có                    100                          

8

Trace BFS theo tầng cho `n = 3` (Cách 2): in tập trạng thái sau mỗi lần bấm để thấy quá trình lớn dần `1 → 4 → 7 → 8`.

In [20]:
def trace_bfs(n, presses):
    m = min(n, 6)
    masks = make_masks(m)
    cur = {(1 << m) - 1}
    print(f"t = 0: {sorted(format(x, f'0{m}b') for x in cur)}  ({len(cur)} trạng thái)")
    for t in range(1, presses + 1):
        cur = {st ^ mk for st in cur for mk in masks}
        print(f"t = {t}: {sorted(format(x, f'0{m}b') for x in cur)}  ({len(cur)} trạng thái)")
    return len(cur)

trace_bfs(3, 4)

t = 0: ['111']  (1 trạng thái)
t = 1: ['000', '010', '101', '110']  (4 trạng thái)
t = 2: ['000', '001', '010', '011', '100', '101', '111']  (7 trạng thái)
t = 3: ['000', '001', '010', '011', '100', '101', '110', '111']  (8 trạng thái)
t = 4: ['000', '001', '010', '011', '100', '101', '110', '111']  (8 trạng thái)


8

## 9. Code nộp LeetCode

Lời giải chuẩn: liệt kê 16 tập nút theo tính chẵn lẻ (Cách 3) — có lý lẽ rõ ràng, không thuộc lòng bảng, `O(1)`. (Cách công thức ở mục 7.3 ngắn hơn và cũng được chấp nhận.) Đúng `class Solution` và signature.

In [21]:
from typing import List

class Solution:
    def flipLights(self, n: int, presses: int) -> int:
        m = min(n, 6)                                   # bóng j và j + 6 luôn giống nhau
        full = (1 << m) - 1                             # tất cả bật
        # mặt nạ 4 nút trên 6 bóng (bit j-1 là bóng j), cắt về m bóng đầu
        ops = [0b111111 & full, 0b101010 & full, 0b010101 & full, 0b001001 & full]
        seen = set()
        for sub in range(16):                           # nút nào bấm số LẺ lần
            k = sub.bit_count()
            if k <= presses and (presses - k) % 2 == 0: # số lần bấm thừa phải đi theo cặp
                st = full
                for b in range(4):
                    if (sub >> b) & 1:
                        st ^= ops[b]
                seen.add(st)
        return len(seen)

sol = Solution()
for n, presses, expected in examples:
    got = sol.flipLights(n, presses)
    print(got, expected, "OK" if got == expected else "SAI")

2 2 OK
3 3 OK
4 4 OK


## 10. Vì sao đúng

**Bước 1 — Mô hình XOR.** Gọi `v_b` là mặt nạ của nút `b`. Bấm nút `b` biến trạng thái `x` thành `x ^ v_b`. Sau một dãy bấm, trạng thái cuối `= full ^ (v_{b1} ^ v_{b2} ^ ...)`. XOR giao hoán/kết hợp và `v ^ v = 0`, nên chỉ phụ thuộc tập `S` các nút bấm **số lẻ** lần. Trạng thái cuối `= full ^ (XOR của v_b, b ∈ S)`.

**Bước 2 — Điều kiện của `S`.** Cần `|S| ≤ presses` vì mỗi nút trong `S` cần ≥ 1 lần bấm. Số lần bấm còn lại `presses - |S|` được chia cho các nút sao cho **mỗi nút chẵn lần thêm** (để không đổi tính chẵn/lẻ), nên phải là số chẵn (ví dụ bấm một nút bất kỳ thêm 2, 4, ... lần). Ngược lại mọi `S` thoả hai điều kiện đều tạo được một dãy bấm đúng `presses` lần. Vậy tập trạng thái = `{ full ^ XOR(S) : S hợp lệ }`. Code làm đúng như thế.

**Bước 3 — Chu kỳ 6.** Mỗi `v_b` có bit `j` bằng bit `j + 6` (kiểm tra ở mục 6), nên mọi trạng thái đạt được có bit `j = bit j + 6`; do đó chỉ cần xét `min(n, 6)` bóng đầu, và hai trạng thái khác nhau trên `n` bóng cũng khác nhau trên `min(n, 6)` bóng đầu (và ngược lại).

**Bước 4 — Công thức.** Nút 1 `= v_2 ^ v_3` (đảo chẵn rồi lẻ = đảo tất cả). Vậy `span{v_1..v_4}` có hạng (rank) tối đa 3 → tối đa `2^3 = 8` trạng thái. Hạng thực tế theo số bóng `m = min(n, 6)`: `m = 1` hạng 1 (2 trạng thái), `m = 2` hạng 2 (4), `m ≥ 3` hạng 3 (8). Số trạng thái khi `presses` đủ lớn đạt đúng `2^rank`; với `presses` nhỏ thì bị hạn chế bởi `|S| ≤ presses`: `presses = 0` chỉ `S = {}` (1 trạng thái); `presses = 1` chỉ `|S| = 1` (tối đa 4 tập); `presses = 2` có `S = {}` và 6 tập hai nút (tối đa 7); từ `presses = 3` có thêm tập 1 nút và 3 nút. Cell dưới tính hạng bằng khử Gauss trên `GF(2)` và so với bảng.

In [22]:
def rank_gf2(vectors):
    # hạng của tập vector bit trên GF(2) bằng khử Gauss (cơ sở 'xor basis')
    basis = []
    for v in vectors:
        for b in basis:
            v = min(v, v ^ b)
        if v:
            basis.append(v)
    return len(basis)

for m in range(1, 8):
    masks = make_masks(min(m, 6))
    r = rank_gf2(masks)
    print(f"m = {m}: hạng = {r} -> tối đa {2 ** r} trạng thái; flipLights(m, 1000) = {Solution().flipLights(m, 1000)}")
    assert Solution().flipLights(m, 1000) == 2 ** r

m = 1: hạng = 1 -> tối đa 2 trạng thái; flipLights(m, 1000) = 2
m = 2: hạng = 2 -> tối đa 4 trạng thái; flipLights(m, 1000) = 4
m = 3: hạng = 3 -> tối đa 8 trạng thái; flipLights(m, 1000) = 8
m = 4: hạng = 3 -> tối đa 8 trạng thái; flipLights(m, 1000) = 8
m = 5: hạng = 3 -> tối đa 8 trạng thái; flipLights(m, 1000) = 8
m = 6: hạng = 3 -> tối đa 8 trạng thái; flipLights(m, 1000) = 8
m = 7: hạng = 3 -> tối đa 8 trạng thái; flipLights(m, 1000) = 8


Tiếp theo kiểm chứng toàn bộ lập luận bằng **oracle thô**: liệt kê *mọi dãy bấm* `product(range(4), repeat=presses)` rồi mô phỏng trực tiếp trên đủ `n` bóng, so với `Solution` cho mọi `n ≤ 8`, `presses ≤ 7`.

In [23]:
def oracle_enumeration(n, presses):
    states = set()
    for seq in product((1, 2, 3, 4), repeat=presses):
        st = tuple([True] * n)
        for b in seq:
            st = press(st, b)
        states.add(st)
    return len(states)

for n in range(1, 9):
    for presses in range(0, 8):
        assert oracle_enumeration(n, presses) == Solution().flipLights(n, presses), (n, presses)
print("oracle liệt kê mọi dãy bấm khớp Solution cho n <= 8, presses <= 7")

oracle liệt kê mọi dãy bấm khớp Solution cho n <= 8, presses <= 7


## 11. Time / Space Complexity

- **Time:** `O(1)` — 16 tập nút, mỗi tập tối đa 4 phép XOR, không phụ thuộc `n` hay `presses`.
- **Space:** `O(1)` — tập trạng thái tối đa 8 phần tử.

So sánh: BFS đủ `n` bóng `O(presses · n)` (hằng số 8·4); BFS 6 bóng `O(presses)`; liệt kê tập nút và công thức `O(1)`.

In [24]:
import time
t0 = time.perf_counter()
a = SolutionSubsets().flipLights(1000, 1000)
t1 = time.perf_counter()
b = SolutionBfsMask().flipLights(1000, 1000)
t2 = time.perf_counter()
c = flip_lights_bfs_full(60, 60)          # BFS đủ bóng chỉ chạy trên cỡ nhỏ
print(a, b, c)
assert a == b == c == 8
print("O(1) và O(presses) đều rất nhanh:", (t2 - t0) < 1)

8 8 8
O(1) và O(presses) đều rất nhanh: True


## 12. Bộ test

Gồm: ví dụ đề, edge case, phản ví dụ cho các cách sai, đối chiếu các cách (và oracle) bằng `assert`, test ngẫu nhiên có seed trong ràng buộc.

In [25]:
sol = Solution()

# 1) Ví dụ đề
assert sol.flipLights(1, 1) == 2
assert sol.flipLights(2, 1) == 3
assert sol.flipLights(3, 1) == 4
print("ví dụ đề OK")

# 2) Edge case
assert sol.flipLights(1, 0) == 1          # không bấm: 1 trạng thái
assert sol.flipLights(1000, 0) == 1
assert sol.flipLights(1, 1000) == 2       # 1 bóng, bấm nhiều
assert sol.flipLights(2, 2) == 4
assert sol.flipLights(2, 1000) == 4
assert sol.flipLights(3, 2) == 7
assert sol.flipLights(3, 3) == 8
assert sol.flipLights(4, 2) == 7
assert sol.flipLights(5, 1) == 4
assert sol.flipLights(6, 1) == 4
assert sol.flipLights(1000, 1) == 4
assert sol.flipLights(1000, 2) == 7
assert sol.flipLights(1000, 1000) == 8
print("edge case OK")

ví dụ đề OK
edge case OK


In [26]:
# 3) Bảng đáp án đầy đủ cho n = 1..8, presses = 0..8: Solution == công thức == BFS mask == liệt kê
solutions = [SolutionBfsMask(), SolutionSubsets(), SolutionFormula(), Solution()]
for n in range(1, 9):
    for presses in range(0, 9):
        expected = flip_lights_bfs_full(n, presses)          # mô phỏng đủ n bóng (cách 1)
        assert flip_lights_bfs_mask(n, presses) == expected
        assert flip_lights_subsets(n, presses) == expected
        assert flip_lights_formula(n, presses) == expected
        for s_ in solutions:
            assert s_.flipLights(n, presses) == expected, (type(s_).__name__, n, presses)
print("bảng 8 x 9: mọi cách khớp BFS mô phỏng đủ bóng")

# 4) Đối chiếu cách 1 với n lớn hơn nhưng presses nhỏ (n tới 30, presses tới 12)
for n in (9, 12, 13, 18, 25, 30):
    for presses in range(0, 13):
        assert flip_lights_bfs_full(n, presses) == Solution().flipLights(n, presses), (n, presses)
print("n tới 30, presses tới 12: khớp")

bảng 8 x 9: mọi cách khớp BFS mô phỏng đủ bóng
n tới 30, presses tới 12: khớp


In [27]:
# 5) Ngẫu nhiên có seed, đúng ràng buộc 1<=n<=1000, 0<=presses<=1000
random.seed(2026)
for _ in range(3000):
    n = random.randint(1, 1000)
    presses = random.randint(0, 1000)
    expected = flip_lights_formula(n, presses)
    for s_ in solutions:
        assert s_.flipLights(n, presses) == expected, (type(s_).__name__, n, presses)
print("3000 cặp ngẫu nhiên: các cách khớp")

# 6) Test tính chất: đơn điệu theo presses (khi tăng 2 lần bấm không giảm), n >= 6 bão hoà, giá trị trong {1,2,3,4,7,8}
for n in range(1, 40):
    prev = 0
    for presses in range(0, 30):
        v = sol.flipLights(n, presses)
        assert v in (1, 2, 3, 4, 7, 8)
        if presses >= 2:
            assert v >= sol.flipLights(n, presses - 2)        # thêm 2 lần bấm không làm mất trạng thái nào
        if n >= 6:
            assert v == sol.flipLights(6, presses)            # bão hoà theo chu kỳ 6
print("test tính chất OK")

3000 cặp ngẫu nhiên: các cách khớp
test tính chất OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cho rằng luôn có `2^4 = 16` trạng thái

Nghĩ "4 nút, mỗi nút bấm hay không" → 16. Nhưng nút 1 = nút 2 + nút 3 nên hai tập khác nhau cho cùng trạng thái, và cũng phải xét `presses`, `n`.

In [28]:
def wrong_16(n, presses):
    return min(16, 2 ** n)                      # SAI

for n, p in [(3, 1), (3, 3), (1000, 1000)]:
    print(f"n={n}, presses={p}: sai {wrong_16(n, p)} | đúng {Solution().flipLights(n, p)}")

n=3, presses=1: sai 8 | đúng 4
n=3, presses=3: sai 8 | đúng 8
n=1000, presses=1000: sai 16 | đúng 8


**Vì sao sai:** hạng chỉ là 3 (nút 1 phụ thuộc nút 2, 3), nên tối đa 8; và `presses` nhỏ còn hạn chế hơn nữa. **Cách nhớ:** đếm *trạng thái* chứ không đếm *cách bấm*.

### Lỗi 2 — Quên trường hợp `presses == 0`

Không bấm gì thì chỉ có trạng thái ban đầu (tất cả bật): đáp án `1`, không phải bảng của `presses ≥ 1`.

In [29]:
def wrong_no_zero(n, presses):
    if n == 1:
        return 2
    if n == 2:
        return 3 if presses == 1 else 4
    if presses == 1:
        return 4
    if presses == 2:
        return 7
    return 8                                    # SAI: presses = 0 rơi vào đây -> 8

print("n=3, presses=0: sai", wrong_no_zero(3, 0), "| đúng", Solution().flipLights(3, 0))
print("n=1, presses=0: sai", wrong_no_zero(1, 0), "| đúng", Solution().flipLights(1, 0))

n=3, presses=0: sai 8 | đúng 1
n=1, presses=0: sai 2 | đúng 1


**Vì sao sai:** không bấm lần nào thì không thể đổi gì. **Quy tắc:** luôn xét `presses == 0` đầu tiên.

### Lỗi 3 — Dùng bảng của `n ≥ 3` cho `n = 1` hoặc `n = 2`

Với `n` nhỏ nhiều nút trở thành giống nhau (ví dụ `n = 1`: nút 2 không làm gì, còn nút 1, 3, 4 cùng đảo đúng một bóng), nên trạng thái ít hơn.

In [30]:
def wrong_big_table(n, presses):
    if presses == 0:
        return 1
    if presses == 1:
        return 4
    if presses == 2:
        return 7
    return 8                                    # SAI: bỏ qua n = 1, 2

for n, p in [(1, 1), (2, 1), (2, 2), (1, 3)]:
    print(f"n={n}, presses={p}: sai {wrong_big_table(n, p)} | đúng {Solution().flipLights(n, p)}")

n=1, presses=1: sai 4 | đúng 2
n=2, presses=1: sai 4 | đúng 3
n=2, presses=2: sai 7 | đúng 4
n=1, presses=3: sai 8 | đúng 2


**Vì sao sai:** với `n = 2` chỉ có 4 trạng thái có thể (`2^2`), không thể có `7` hay `8`. **Quy tắc:** trạng thái không bao giờ vượt `2^min(n, 3)` (vì hạng ≤ 3, hạng ≤ số bóng).

### Lỗi 4 — Bỏ qua điều kiện chẵn/lẻ của số lần bấm

Chỉ kiểm tra `|S| ≤ presses` mà không kiểm tra `(presses - |S|)` chẵn. Khi đó ta cho phép nhầm tập rỗng khi `presses = 1` (không bấm gì mà lại phải bấm đúng 1 lần) và tập một nút khi `presses = 2`.

In [31]:
def wrong_no_parity(n, presses):
    m = min(n, 6)
    masks = make_masks(m)
    full = (1 << m) - 1
    seen = set()
    for sub in range(16):
        if sub.bit_count() <= presses:           # SAI: thiếu điều kiện (presses - k) chẵn
            st = full
            for b in range(4):
                if (sub >> b) & 1:
                    st ^= masks[b]
            seen.add(st)
    return len(seen)

for n, p in [(3, 1), (3, 2), (3, 3)]:
    print(f"n={n}, presses={p}: sai {wrong_no_parity(n, p)} | đúng {Solution().flipLights(n, p)}")

n=3, presses=1: sai 5 | đúng 4
n=3, presses=2: sai 8 | đúng 7
n=3, presses=3: sai 8 | đúng 8


**Vì sao sai:** với `presses = 1`, tập rỗng (không bấm nút nào) không thể đạt được vì bắt buộc bấm đúng 1 lần; tương tự với `presses = 2` các tập một nút cần 1 lần bấm lẻ — không thể thừa 1 lần lẻ. Điều kiện `(presses - |S|) % 2 == 0` bắt buộc.

### Lỗi 5 — Mô phỏng đủ `n = 1000` bóng với `presses = 1000`

Mô phỏng từng bóng cho từng lần bấm là hàng chục triệu thao tác. Dù Python chạy được trong vài chục giây, judge sẽ giới hạn thời gian. Hãy giảm bài toán trước (chu kỳ 6) hoặc tìm quy luật.

In [32]:
import time
t0 = time.perf_counter()
r = flip_lights_bfs_full(300, 60)      # n = 300, 60 lần bấm: vẫn tạm chạy được nhưng đã chậm hơn hẳn
t1 = time.perf_counter()
r2 = Solution().flipLights(300, 60)
t2 = time.perf_counter()
print("cùng đáp án:", r == r2)
print("mô phỏng đủ bóng chậm hơn cách O(1):", (t1 - t0) > (t2 - t1))

cùng đáp án: True
mô phỏng đủ bóng chậm hơn cách O(1): True


**Cách nhớ:** khi đề cho `n` rất lớn nhưng thao tác chỉ phụ thuộc `j mod k`, hãy thu về **một chu kỳ**.

## 14. Mẹo & nhận dạng pattern

- **"Bật/tắt, đảo trạng thái, thực hiện đúng k lần, đếm trạng thái"** → mô hình XOR. Hai lần đảo huỷ nhau → chỉ cần **tính chẵn/lẻ**; lập tập `S` nút bấm lẻ lần.
- **Điều kiện đạt được:** `|S| ≤ k` và `k - |S|` chẵn (số lần thừa đi theo cặp). Nếu có một **cặp nút giao hoán cho cùng hiệu ứng** (như nút 1 = nút 2 ^ nút 3) thì số trạng thái giảm vì các tập khác nhau trùng.
- **Chu kỳ theo `lcm`:** hiệu ứng chỉ phụ thuộc `j mod 2`, `j mod 3` → chu kỳ `lcm = 6`. Rút `n` lớn về `min(n, 6)`.
- **Đáp án nhỏ nhưng input lớn** → chắc chắn có quy luật/bão hoà; hãy **in bảng** các trường hợp nhỏ rồi quan sát (như mục 7.3).
- **Hạng trên `GF(2)`:** số trạng thái tối đa là `2^rank` của các mặt nạ.
- Bài họ hàng: Bulb Switcher (319), Bulb Switcher III (1375), Flip Game II (294), Reverse bits/bitmask.

## 15. Biến thể

### 15.1. Liệt kê chính các trạng thái (không chỉ đếm)

Trả về danh sách các trạng thái dạng chuỗi `on/off` cho `n` bóng. Dùng chu kỳ 6 để nhân bản về đủ `n` bóng.

In [33]:
def reachable_states(n, presses):
    m = min(n, 6)
    masks = make_masks(m)
    full = (1 << m) - 1
    seen = set()
    for sub in range(16):
        k = sub.bit_count()
        if k <= presses and (presses - k) % 2 == 0:
            st = full
            for b in range(4):
                if (sub >> b) & 1:
                    st ^= masks[b]
            seen.add(st)
    # nhân bản chu kỳ: bóng thứ j (đếm từ 0) lấy bit j % 6
    out = []
    for st in sorted(seen):
        bulbs = ["on" if (st >> (j % 6)) & 1 else "off" for j in range(n)]
        out.append(bulbs)
    return out

for row in reachable_states(3, 1):
    print(row)
print(len(reachable_states(8, 3)))        # 8 trạng thái trên 8 bóng

['off', 'off', 'off']
['off', 'on', 'off']
['on', 'off', 'on']
['off', 'on', 'on']
8


### 15.2. Kiểm tra một trạng thái đích có đạt được không

Cho `n, presses` và mảng `target` (độ dài `n`, `True` = bật). Trạng thái đích đạt được khi **nằm trong** tập trạng thái đạt được. Ta dựng `target` về dạng mặt nạ của `min(n, 6)` bóng đầu và kiểm tra chu kỳ (bóng `j` phải bằng bóng `j + 6`).

In [34]:
def reachable(n, presses, target):
    if len(target) != n:
        raise ValueError("target phải có độ dài n")
    # target phải thoả chu kỳ 6
    for j in range(6, n):
        if target[j] != target[j - 6]:
            return False
    m = min(n, 6)
    want = sum(1 << i for i in range(m) if target[i])
    masks = make_masks(m)
    full = (1 << m) - 1
    for sub in range(16):
        k = sub.bit_count()
        if k <= presses and (presses - k) % 2 == 0:
            st = full
            for b in range(4):
                if (sub >> b) & 1:
                    st ^= masks[b]
            if st == want:
                return True
    return False

print(reachable(3, 1, [False, False, False]))      # True  (nút 1)
print(reachable(3, 1, [True, True, True]))         # False (cần 0 lần bấm, nhưng phải bấm đúng 1)
print(reachable(3, 2, [True, True, True]))         # True  (bấm cùng một nút hai lần)

True
False
True


### 15.3. Thêm nút khác (tổng quát hoá)

Tổng quát: có `K` nút, mỗi nút là một vị từ trên nhãn `j`. Quy trình giống hệt: xác định **chu kỳ** `L = lcm` các chu kỳ, trạng thái xác định bởi `min(n, L)` bóng đầu, liệt kê `2^K` tập nút theo tính chẵn/lẻ. Dưới đây là hàm tổng quát (có kiểm chứng bằng BFS mô phỏng đủ bóng cho một bộ nút mới: thêm nút 5 đảo bóng có `j mod 4 == 0`).

In [35]:
from math import lcm

def count_states_general(n, presses, preds, period):
    # preds: list các hàm j -> bool (j từ 1); period: chu kỳ chung của mọi nút
    m = min(n, period)
    masks = [sum(1 << (j - 1) for j in range(1, m + 1) if p(j)) for p in preds]
    full = (1 << m) - 1
    K = len(preds)
    seen = set()
    for sub in range(1 << K):
        k = sub.bit_count()
        if k <= presses and (presses - k) % 2 == 0:
            st = full
            for b in range(K):
                if (sub >> b) & 1:
                    st ^= masks[b]
            seen.add(st)
    return len(seen)

def brute_general(n, presses, preds):
    cur = {tuple([True] * n)}
    for _ in range(presses):
        cur = {tuple((not on) if p(j + 1) else on for j, on in enumerate(st)) for st in cur for p in preds}
    return len(cur)

original = [lambda j: True, lambda j: j % 2 == 0, lambda j: j % 2 == 1, lambda j: j % 3 == 1]
for n in range(1, 8):
    for presses in range(0, 7):
        assert count_states_general(n, presses, original, 6) == Solution().flipLights(n, presses)

extended = original + [lambda j: j % 4 == 0]
period = lcm(2, 3, 4)                      # 12
for n in range(1, 15):
    for presses in range(0, 7):
        assert count_states_general(n, presses, extended, period) == brute_general(n, presses, extended), (n, presses)
print("tổng quát hoá khớp BFS đủ bóng; chu kỳ mới =", period)
print("5 nút, n = 12, presses = 1000:", count_states_general(12, 1000, extended, period))

tổng quát hoá khớp BFS đủ bóng; chu kỳ mới = 12
5 nút, n = 12, presses = 1000: 16


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS đủ `n` bóng | Mô phỏng từng trạng thái qua từng lần bấm | `O(presses · n)` (hằng ≈ 32) | `O(n)` | Oracle với `n, presses` nhỏ |
| 2. BFS 6 bóng bằng bitmask | Chu kỳ 6 + XOR mặt nạ | `O(presses)` | `O(64)` | Khi chưa nhận ra tính chẵn/lẻ nhưng đã thấy chu kỳ |
| 3. Liệt kê 16 tập nút | Chỉ chẵn/lẻ số lần bấm; `|S| ≤ presses`, cùng tính chẵn lẻ | `O(1)` | `O(8)` | **Lời giải chuẩn nộp LeetCode**; tổng quát hoá được |
| 4. Công thức theo trường hợp | Bảng `(n, presses)` rút từ cách 3 | `O(1)` | `O(1)` | Ngắn nhất; cần chứng minh/kiểm chứng bảng |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy kiểm tra:

```python
print(Solution().flipLights(1, 0))
print(Solution().flipLights(2, 2))
print(Solution().flipLights(1000, 1000))
```

In [36]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `n = 3` và `presses = 2`: liệt kê các tập nút `S` hợp lệ (`|S| ≤ 2` và `2 - |S|` chẵn), tính trạng thái của từng tập (mặt nạ: nút 1 = `111`, nút 2 = `010`, nút 3 = `101`, nút 4 = `001`; bắt đầu `111`), rồi đếm số trạng thái khác nhau. Điền vào các biến.

In [37]:
# Bài 2: điền kết quả tính tay
hand_valid_subsets = None    # số tập S hợp lệ
hand_answer = None           # số trạng thái khác nhau

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm công thức theo trường hợp.

In [38]:
def exercise_3_flip_lights(n: int, presses: int) -> int:
    # TODO 1: presses == 0 -> 1 trạng thái
    # TODO 2: n == 1 -> 2
    # TODO 3: n == 2 -> presses == 1 thì 3, ngược lại 4
    # TODO 4: n >= 3 -> presses == 1: 4; presses == 2: 7; còn lại: 8
    return None


# assert exercise_3_flip_lights(1, 1) == 2
# assert exercise_3_flip_lights(2, 1) == 3
# assert exercise_3_flip_lights(3, 1) == 4
# assert exercise_3_flip_lights(3, 0) == 1
# assert exercise_3_flip_lights(1000, 1000) == 8

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, hãy tự viết `exercise_4_subsets(n, presses)`: dùng mặt nạ bit cho 4 nút trên `min(n, 6)` bóng, duyệt 16 tập nút, chỉ giữ tập có `k <= presses` và `(presses - k) % 2 == 0`, đếm số trạng thái khác nhau.

In [39]:
def exercise_4_subsets(n: int, presses: int) -> int:
    # TODO
    pass


# assert exercise_4_subsets(3, 1) == 4
# assert exercise_4_subsets(3, 2) == 7
# assert exercise_4_subsets(2, 5) == 4
# assert exercise_4_subsets(1, 0) == 1

### Bài 5 — Mini challenge: đạt được trạng thái đích? (khó)

Viết `exercise_5_reachable(n, presses, target)` (`target` là danh sách `n` giá trị `True/False`, `True` = bật) trả `True` nếu sau **đúng** `presses` lần bấm có thể ra đúng `target`. Gợi ý: `target` phải thoả `target[j] == target[j - 6]` với `j ≥ 6`; kiểm tra trong tập trạng thái đạt được.

```python
exercise_5_reachable(3, 1, [False, False, False]) -> True
exercise_5_reachable(3, 1, [True, True, True])    -> False
exercise_5_reachable(3, 2, [True, True, True])    -> True
exercise_5_reachable(7, 3, [True, False, True, True, False, True, False]) -> hãy tự kiểm tra
```

In [40]:
def exercise_5_reachable(n: int, presses: int, target: List[bool]) -> bool:
    # TODO
    pass


# kết quả được kiểm tra ở mục 19 bằng BFS mô phỏng đủ bóng

## 18. Gợi ý 3 mức

**Bài 3 / Bài 4**

- *Mức 1 (nhẹ):* Bấm một nút hai lần huỷ nhau. Vậy chỉ cần biết nút nào được bấm **số lẻ lần**.
- *Mức 2 (vừa):* Một tập `S` nút (bấm lẻ lần) đạt được với đúng `presses` lần khi `|S| ≤ presses` và `presses - |S|` chẵn. Chỉ xét `min(n, 6)` bóng đầu.
- *Mức 3 (gần đáp án):* `full = (1 << m) - 1`; `ops = [0b111111 & full, 0b101010 & full, 0b010101 & full, 0b001001 & full]`; với mỗi `sub` hợp lệ, `st = full` rồi `st ^= ops[b]` cho mỗi bit `b` của `sub`; thêm vào `set`; trả về `len`. Công thức: xét `presses == 0`, `n == 1`, `n == 2`, rồi `presses == 1, 2, ≥ 3`.

**Bài 5**

- *Mức 1:* Phần nào của bài đã cho biết tập trạng thái đạt được?
- *Mức 2:* Biểu diễn `target[0..m-1]` thành mặt nạ `want`; kiểm tra chu kỳ 6 cho các bóng còn lại.
- *Mức 3:* Trả `True` nếu `want` thuộc tập `{full ^ XOR(S) : S hợp lệ}` và `target` thoả chu kỳ 6; còn lại `False`.

## 19. Đáp án tham khảo

### Đáp án Bài 1

- `flipLights(1, 0)` → `1` (không bấm).
- `flipLights(2, 2)` → `4`.
- `flipLights(1000, 1000)` → `8`.

In [41]:
assert Solution().flipLights(1, 0) == 1
assert Solution().flipLights(2, 2) == 4
assert Solution().flipLights(1000, 1000) == 8
print("Bài 1 OK")

Bài 1 OK


### Đáp án Bài 2

Tập hợp lệ (`presses = 2`): `S = {}` (`|S| = 0`, `2 - 0` chẵn) và mọi tập hai nút (`|S| = 2`): `C(4, 2) = 6` tập. Tổng `7` tập hợp lệ.

Trạng thái (bắt đầu `111`): `{}` → `111`; `{1,2}` → `111 ^ 111 ^ 010 = 010`; `{1,3}` → `111 ^ 111 ^ 101 = 101`; `{1,4}` → `111 ^ 111 ^ 001 = 001`; `{2,3}` → `111 ^ 010 ^ 101 = 000`; `{2,4}` → `111 ^ 010 ^ 001 = 100`; `{3,4}` → `111 ^ 101 ^ 001 = 011`. Bảy trạng thái `111, 010, 101, 001, 000, 100, 011` đều khác nhau → đáp án `7`.

In [42]:
hand_valid_subsets, hand_answer = 7, 7
m_ = 3
masks_ = make_masks(m_)
full_ = (1 << m_) - 1
valid = [sub for sub in range(16) if sub.bit_count() <= 2 and (2 - sub.bit_count()) % 2 == 0]
assert len(valid) == hand_valid_subsets
states_ = set()
for sub in valid:
    st = full_
    for b in range(4):
        if (sub >> b) & 1:
            st ^= masks_[b]
    states_.add(st)
assert len(states_) == hand_answer == Solution().flipLights(3, 2) == oracle_enumeration(3, 2)
print("Bài 2 OK; các trạng thái:", sorted(format(x, "03b") for x in states_))

Bài 2 OK; các trạng thái: ['000', '001', '010', '011', '100', '101', '111']


### Đáp án Bài 3

In [43]:
def exercise_3_solution(n: int, presses: int) -> int:
    if presses == 0:
        return 1
    if n == 1:
        return 2
    if n == 2:
        return 3 if presses == 1 else 4
    if presses == 1:
        return 4
    if presses == 2:
        return 7
    return 8

assert exercise_3_solution(1, 1) == 2
assert exercise_3_solution(2, 1) == 3
assert exercise_3_solution(3, 1) == 4
assert exercise_3_solution(3, 0) == 1
assert exercise_3_solution(1000, 1000) == 8
for n in range(1, 40):
    for presses in range(0, 20):
        assert exercise_3_solution(n, presses) == Solution().flipLights(n, presses)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [44]:
def exercise_4_solution(n: int, presses: int) -> int:
    m = min(n, 6)
    full = (1 << m) - 1
    ops = [0b111111 & full, 0b101010 & full, 0b010101 & full, 0b001001 & full]
    seen = set()
    for sub in range(16):
        k = sub.bit_count()
        if k <= presses and (presses - k) % 2 == 0:
            st = full
            for b in range(4):
                if (sub >> b) & 1:
                    st ^= ops[b]
            seen.add(st)
    return len(seen)

assert exercise_4_solution(3, 1) == 4
assert exercise_4_solution(3, 2) == 7
assert exercise_4_solution(2, 5) == 4
assert exercise_4_solution(1, 0) == 1
for n in range(1, 30):
    for presses in range(0, 15):
        assert exercise_4_solution(n, presses) == flip_lights_bfs_full(n, presses)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

Với `exercise_5_reachable(7, 3, [True, False, True, True, False, True, False])`: bóng 7 (`j = 6`) phải bằng bóng 1 (`j = 0`): `False` ≠ `True` → vi phạm chu kỳ 6 → `False`. Cell dưới đối chiếu với BFS mô phỏng đủ bóng trên mọi trạng thái đích có thể có.

In [45]:
def exercise_5_solution(n: int, presses: int, target: List[bool]) -> bool:
    for j in range(6, n):
        if target[j] != target[j - 6]:
            return False
    m = min(n, 6)
    full = (1 << m) - 1
    ops = [0b111111 & full, 0b101010 & full, 0b010101 & full, 0b001001 & full]
    want = sum(1 << i for i in range(m) if target[i])
    for sub in range(16):
        k = sub.bit_count()
        if k <= presses and (presses - k) % 2 == 0:
            st = full
            for b in range(4):
                if (sub >> b) & 1:
                    st ^= ops[b]
            if st == want:
                return True
    return False

assert exercise_5_solution(3, 1, [False, False, False]) is True
assert exercise_5_solution(3, 1, [True, True, True]) is False
assert exercise_5_solution(3, 2, [True, True, True]) is True
assert exercise_5_solution(7, 3, [True, False, True, True, False, True, False]) is False

# đối chiếu với BFS mô phỏng đủ bóng: tập trạng thái thật sự đạt được
def real_states(n, presses):
    cur = {tuple([True] * n)}
    for _ in range(presses):
        cur = {press(st, b) for st in cur for b in (1, 2, 3, 4)}
    return cur

for n in (1, 2, 3, 4, 7):
    for presses in range(0, 6):
        reach = real_states(n, presses)
        for target in product((False, True), repeat=n):
            assert exercise_5_solution(n, presses, list(target)) == (tuple(target) in reach), (n, presses, target)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def flipLights(self, n: int, presses: int) -> int:
        m = min(n, 6)                                    # chu kỳ lcm(2, 3) = 6
        full = (1 << m) - 1                              # tất cả bật
        ops = [0b111111 & full, 0b101010 & full, 0b010101 & full, 0b001001 & full]
        seen = set()
        for sub in range(16):                            # nút nào bấm LẺ lần
            k = sub.bit_count()
            if k <= presses and (presses - k) % 2 == 0:  # số lần thừa phải đi theo cặp
                st = full
                for b in range(4):
                    if (sub >> b) & 1:
                        st ^= ops[b]
                seen.add(st)
        return len(seen)
```

Câu thần chú: **"Bấm hai lần huỷ nhau; chỉ xét tập nút bấm lẻ lần, cùng tính chẵn lẻ với `presses` và không nhiều hơn `presses`; sáu bóng đầu đại diện tất cả."**

### Checklist trước khi submit

- [ ] Đã rút `n` về `min(n, 6)` (chu kỳ 6).
- [ ] Điều kiện tập nút: `k <= presses` **và** `(presses - k) % 2 == 0`.
- [ ] Đếm **trạng thái khác nhau** bằng `set`, không đếm số tập nút.
- [ ] Đã thử `presses = 0` (đáp án `1`), `n = 1`, `n = 2`, `n ≥ 3`, `presses = 1, 2, 3`, `n = presses = 1000`.
- [ ] Đã đối chiếu BFS mô phỏng đủ bóng và oracle liệt kê mọi dãy bấm trên input nhỏ.
- [ ] Skeleton đúng: `class Solution`, method `flipLights(self, n: int, presses: int) -> int`.